# E-commerce Product Analysis Project

The following analysis is performed on dataset from Kaggle.

##### Credits:

https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop

https://rees46.com/

**Project goal is to analyze user behavior in an e-commerce cosmetics shop and identify factors associated with successful purchases.**

## Business Context

The dataset contains user event history from an online cosmetics store.

Users interact with products through several types of events:

view — product viewed;
cart — product added to cart;
remove_from_cart — product removed from cart;
purchase — product purchased.

The business wants to understand:

*How do users move through the shopping funnel, where are potential customers lost, and what user/product characteristics are associated with a higher probability of purchase?*

The final goal is **to identify product opportunities that could potentially improve purchase conversion.**

## Import Required Python Libraries

In [3]:
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

## Dataset Installation

In [4]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = kagglehub.dataset_download(
    "mkechinov/ecommerce-events-history-in-cosmetics-shop",
    output_dir=str(data_dir.resolve()),
)

print(f"Dataset downloaded to: {dataset_path}")

Dataset downloaded to: /Users/likanikolaidi/ecommerce-product-analysis/data


## Dataset Properties Description

**event_time**	Time when event happened at (in UTC).

**event_type**	Four kinds of event: purchase, cart, view, remove_from_cart.

**product_id**	ID of a product

**category_id**	Product's category ID

**category_code**	Product's category taxonomy (code name) if it was possible to make it. Usually present for meaningful categories and skipped for different kinds of accessories.

**brand**	Downcased string of brand name. Can be missed.

**price**	Float price of a product. Present.

**user_id**	Permanent user ID.

**user_session**  Temporary user's session ID. Same for each user's session. Is changed every time user come back to online store from a long pause.

## Data Inspection

In [5]:
csv_files = sorted(data_dir.rglob("*.csv"))
for file in csv_files:
    print(file.name, f"{file.stat().st_size / 1e9:.2f} GB")

2019-Dec.csv 0.42 GB
2019-Nov.csv 0.55 GB
2019-Oct.csv 0.48 GB
2020-Feb.csv 0.49 GB
2020-Jan.csv 0.50 GB


In [6]:
sample = pd.read_csv(csv_files[0], nrows=100_000)
sample.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-01 00:00:00 UTC,remove_from_cart,5712790,1487580005268456287,NaN,f.o.x,6.27,576802932,51d85cb0-897f-48d2-918b-ad63965c12dc
1,2019-12-01 00:00:00 UTC,view,5764655,1487580005411062629,NaN,cnd,29.05,412120092,8adff31e-2051-4894-9758-224bfa8aec18
2,2019-12-01 00:00:02 UTC,cart,4958,1487580009471148064,NaN,runail,1.19,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554
3,2019-12-01 00:00:05 UTC,view,5848413,1487580007675986893,NaN,freedecor,0.79,348405118,722ffea5-73c0-4924-8e8f-371ff8031af4
4,2019-12-01 00:00:07 UTC,view,5824148,1487580005511725929,NaN,NaN,5.56,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5


In [11]:
# inspecting the sample
print(sample.shape)
print(sample.columns)
print(sample.dtypes)
print(sample.isna().sum())

(100000, 9)
Index(['event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
event_time           str
event_type           str
product_id         int64
category_id        int64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
dtype: object
event_time           0
event_type           0
product_id           0
category_id          0
category_code    98309
brand            43558
price                0
user_id              0
user_session        43
dtype: int64


For the computational resource economy the whole exploratory pipline will be first set up for one month data only

In [ ]:
oct_df = pd.read_csv("data/2019-Oct.csv")
oct_df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,cart,5773203,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
1,2019-10-01 00:00:03 UTC,cart,5773353,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
2,2019-10-01 00:00:07 UTC,cart,5881589,2151191071051219817,NaN,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3,2019-10-01 00:00:07 UTC,cart,5723490,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
4,2019-10-01 00:00:15 UTC,cart,5881449,1487580013522845895,NaN,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9


## Data Quality Checks

Includes duplicates, date range, event types, and counts of unique users, sessions, products, categories, and brands.

In [18]:
df = oct_df

In [19]:
EXPECTED_COLUMNS = {
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
}

EXPECTED_EVENT_TYPES = {
    "view",
    "cart",
    "remove_from_cart",
    "purchase",
}


def run_data_quality_checks(df):
    missing_columns = EXPECTED_COLUMNS - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    event_time = pd.to_datetime(
        df["event_time"],
        errors="coerce",
        utc=True,
    )

    overview = pd.Series({
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "invalid_event_times": event_time.isna().sum(),
        "date_min": event_time.min(),
        "date_max": event_time.max(),
        "unique_users": df["user_id"].nunique(dropna=True),
        "unique_sessions": df["user_session"].nunique(dropna=True),
        "unique_products": df["product_id"].nunique(dropna=True),
        "unique_categories": df["category_id"].nunique(dropna=True),
        "unique_brands": df["brand"].nunique(dropna=True),
        "non_positive_prices": df["price"].le(0).sum(),
    }, name="value")

    missing_values = (
        pd.DataFrame({
            "missing_count": df.isna().sum(),
            "missing_percent": df.isna().mean().mul(100),
        })
        .sort_values("missing_percent", ascending=False)
    )

    data_types = df.dtypes.rename("data_type").to_frame()

    event_distribution = (
        df["event_type"]
        .value_counts(dropna=False)
        .rename_axis("event_type")
        .to_frame("event_count")
    )
    event_distribution["percent"] = (
        event_distribution["event_count"]
        .div(len(df))
        .mul(100)
    )

    unexpected_events = sorted(
        set(df["event_type"].dropna().unique())
        - EXPECTED_EVENT_TYPES
    )

    return {
        "overview": overview,
        "data_types": data_types,
        "missing_values": missing_values,
        "event_distribution": event_distribution,
        "unexpected_events": unexpected_events,
    }

In [20]:
quality_report = run_data_quality_checks(df)

display(quality_report["overview"])
display(quality_report["data_types"])
display(quality_report["missing_values"])
display(quality_report["event_distribution"])

print(
    "Unexpected event types:",
    quality_report["unexpected_events"],
)

rows                                     4102283
columns                                        9
duplicate_rows                            213155
invalid_event_times                            0
date_min               2019-10-01 00:00:00+00:00
date_max               2019-10-31 23:59:54+00:00
unique_users                              399664
unique_sessions                           873960
unique_products                            41899
unique_categories                            490
unique_brands                                240
non_positive_prices                         6086
Name: value, dtype: object

,data_type
event_time,str
event_type,str
product_id,int64
category_id,int64
category_code,str
brand,str
price,float64
user_id,int64
user_session,str


,missing_count,missing_percent
category_code,4034806,98.355135
brand,1659261,40.447258
user_session,637,0.015528
event_time,0,0.000000
event_type,0,0.000000
category_id,0,0.000000
product_id,0,0.000000
price,0,0.000000
user_id,0,0.000000


,event_count,percent
event_type,,
view,1862164,45.393358
cart,1232385,30.041443
remove_from_cart,762110,18.577704
purchase,245624,5.987495


Unexpected event types: []


### Investigating questionable records

These records' analysis include duplicate rows (which can legitimately occur) and invalid (negative) prices.

In [21]:
duplicate_mask = df.duplicated(keep=False)

display(
    df.loc[duplicate_mask, "event_type"]
    .value_counts()
    .to_frame("rows_in_duplicate_groups")
)

display(
    df.loc[duplicate_mask]
    .sort_values(
        ["user_id", "user_session", "product_id", "event_time"]
    )
    .head(30)
)

,rows_in_duplicate_groups
event_type,
remove_from_cart,363653
cart,53670
purchase,686
view,228


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
137746,2019-10-01 22:06:07 UTC,remove_from_cart,5739036,1487580008246412266,NaN,kapous,3.32,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb
137748,2019-10-01 22:06:07 UTC,remove_from_cart,5739036,1487580008246412266,NaN,kapous,3.32,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb
137684,2019-10-01 22:05:27 UTC,remove_from_cart,5812138,1487580013413793985,NaN,art-visage,4.68,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb
137685,2019-10-01 22:05:27 UTC,remove_from_cart,5812138,1487580013413793985,NaN,art-visage,4.68,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb
266253,2019-10-02 14:43:55 UTC,remove_from_cart,5709126,1487580011752849537,NaN,elskin,4.21,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad
266256,2019-10-02 14:43:55 UTC,remove_from_cart,5709126,1487580011752849537,NaN,elskin,4.21,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad
266393,2019-10-02 14:44:45 UTC,remove_from_cart,5823668,1487580008246412266,NaN,kapous,7.60,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad
266396,2019-10-02 14:44:45 UTC,remove_from_cart,5823668,1487580008246412266,NaN,kapous,7.60,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad
3535082,2019-10-27 16:00:04 UTC,remove_from_cart,5807805,1487580005713052531,NaN,ingarden,4.44,10280338,fbacb4a3-2141-4f43-b9e5-97c911c7e770
3535083,2019-10-27 16:00:04 UTC,remove_from_cart,5807805,1487580005713052531,NaN,ingarden,4.44,10280338,fbacb4a3-2141-4f43-b9e5-97c911c7e770


In [33]:
invalid_prices = df.loc[df["price"] <= 0]

display(invalid_prices["price"].describe())
display(invalid_prices["event_type"].value_counts())
display(invalid_prices.sample(5))
display(invalid_prices.loc[invalid_prices["price"] <= -20])

count    6086.000000
mean       -0.091285
std         1.906700
min       -79.370000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.000000
Name: price, dtype: float64

event_type
view                5372
cart                 540
remove_from_cart     154
purchase              20
Name: count, dtype: int64

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
3954463,2019-10-30 18:52:46 UTC,view,5892679,1487580008263189483,NaN,NaN,0.0,504242074,42fb4e31-ec57-45fb-9c98-20c6b6b4d14e
3206374,2019-10-24 18:10:36 UTC,view,5896449,1487580007675986893,NaN,NaN,0.0,500771270,6c977b05-453a-494e-927d-348a556753a8
2848762,2019-10-22 05:19:27 UTC,view,5894508,1487580011702517887,NaN,NaN,0.0,562817002,ec305a0b-4133-49b5-9e67-6403691883ce
3803063,2019-10-29 15:17:37 UTC,view,5891052,1783999068909863670,NaN,NaN,0.0,551119191,698dbbfd-6611-4974-9ac4-46a78345551e
3182956,2019-10-24 14:35:21 UTC,view,5896719,1487580007675986893,NaN,NaN,0.0,558605370,0f88922b-e94f-4890-b4bc-bf4ef89335ef


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
112860,2019-10-01 19:10:56 UTC,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,552507528,dcdd60c6-1a70-442d-bfb2-0252879054ad
436918,2019-10-03 17:37:04 UTC,purchase,5716859,1487580014042939619,NaN,NaN,-47.62,555414763,479149eb-1807-4178-8f6b-87c642350735
1519246,2019-10-11 10:27:19 UTC,purchase,5716859,1487580014042939619,NaN,NaN,-47.62,543647038,85e11d74-6583-4eab-b50c-ff86dbb25d97
1783312,2019-10-13 16:46:01 UTC,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,559820267,f178c995-f004-4040-b26d-b1cca0f9657d
1924072,2019-10-14 17:33:24 UTC,purchase,5716861,1487580014042939619,NaN,NaN,-79.37,541122983,b60f777d-afca-4299-8548-273b810d6130
2143762,2019-10-16 11:41:06 UTC,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,461943726,beefa8fb-a3d1-48ca-97e9-9c95b05ee997
3135434,2019-10-24 08:10:06 UTC,purchase,5716859,1487580014042939619,NaN,NaN,-47.62,545859098,95ed1875-cb64-443d-90bb-17d0dc596c10
3502423,2019-10-27 11:24:32 UTC,purchase,5716859,1487580014042939619,NaN,NaN,-47.62,564627373,81d564e9-55b3-4929-9132-6353cab08a97
3608726,2019-10-28 07:41:54 UTC,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,385939606,2255291d-8060-4ce9-934c-458a3bd0b944
3798007,2019-10-29 14:34:24 UTC,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,554081558,4b84a8f9-97d7-4eb1-b120-7b34ddfe9a85


This analysis provides following conclusions:

- The concentration of duplicate rows in cart-removal events suggests these may represent quantity changes or repeated actions rather than random file duplication. Because timestamps only have one-second precision, two legitimate events can also appear identical.

- Zero-price rows often also lack brand/category information, suggesting incomplete product-catalog records. Negative purchase prices are more concerning, but the dataset does not provide enough evidence to classify them as returns, discounts, or errors and the amount of these rows is relatively small.

## Cleaning Pipeline

In [37]:
def clean_event_data(raw_df, drop_exact_duplicates=False):
    clean_df = raw_df.copy()

    # Parse timestamps consistently as UTC
    clean_df["event_time"] = pd.to_datetime(
        clean_df["event_time"],
        errors="coerce",
        utc=True,
    )

    # Standardize text values
    text_columns = ["event_type", "category_code", "brand", "user_session"]

    for column in text_columns:
        clean_df[column] = (
            clean_df[column]
            .astype("string")
            .str.strip()
        )
        clean_df[column] = clean_df[column].mask(
            clean_df[column].eq(""),
            pd.NA,
        )

    clean_df["event_type"] = clean_df["event_type"].str.lower()
    clean_df["brand"] = clean_df["brand"].str.lower()
    clean_df["category_code"] = clean_df["category_code"].str.lower()

    # Validate essential fields
    essential_columns = [
        "event_time",
        "event_type",
        "product_id",
        "category_id",
        "price",
        "user_id",
    ]

    clean_df = clean_df.dropna(subset=essential_columns)

    # Flag valid prices
    clean_df["is_valid_price"] = clean_df["price"] > 0

    # Useful time fields for later aggregation
    clean_df["event_date"] = clean_df["event_time"].dt.floor("D")
    clean_df["event_month"] = (
    clean_df["event_time"]
    .dt.tz_localize(None)
    .dt.to_period("M")
)

    # Duplicate removal remains optional
    if drop_exact_duplicates:
        clean_df = clean_df.drop_duplicates(ignore_index=True)

    return clean_df

In [38]:
clean_df = clean_event_data(
    df,
    drop_exact_duplicates=False,
)

clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4102283 entries, 0 to 4102282
Data columns (total 12 columns):
 #   Column          Dtype              
---  ------          -----              
 0   event_time      datetime64[us, UTC]
 1   event_type      string             
 2   product_id      int64              
 3   category_id     int64              
 4   category_code   string             
 5   brand           string             
 6   price           float64            
 7   user_id         int64              
 8   user_session    string             
 9   is_valid_price  bool               
 10  event_date      datetime64[us, UTC]
 11  event_month     period[M]          
dtypes: bool(1), datetime64[us, UTC](2), float64(1), int64(3), period[M](1), string(4)
memory usage: 348.2 MB
